<a href="https://colab.research.google.com/github/candreoliuoc/tfg-gnn-pagerank/blob/main/notebooks/T3_T4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# I import os. I mount my Google Drive and I create folder TFG if it's possible
# I create 3 subfolders: raw, data and figures. I will save files in there.

import os

from google.colab import drive
drive.mount("/content/drive")
BASE= "/content/drive/MyDrive/TFG"
os.makedirs(BASE, exist_ok=True)

RAW_DIR=f"{BASE}/raw"
DATA_DIR=f"{BASE}/data"
FIG_DIR=f"{BASE}/figures"
for d in (RAW_DIR, DATA_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

print("Files will be saved in", os.path.abspath(BASE))

Mounted at /content/drive
Files will be saved in /content/drive/MyDrive/TFG


In [ ]:
# I import some libraries I will need later

import gzip, re, time, urllib.request, random
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# And I configure some global properties:

pd.options.display.float_format = "{:,.6g}".format


In [ ]:
# To download the files from SNAP, I configure its url and the name of the files
# that I will download: one for the edges and nodes definitions, one for their
# names and one for their categories.
# In a loop, I download these files and I save them in folder raw
# In a second loop, I gunzip each file and show 3 lines, just to have an idea.

SNAP_URL = "https://snap.stanford.edu/data/"
files = {"edges": "wiki-topcats.txt.gz",
         "categories": "wiki-topcats-categories.txt.gz",
         "names": "wiki-topcats-page-names.txt.gz"}

for f in files.values():
  path= os.path.join(RAW_DIR, f)
  if not os.path.exists(path):
    print("Downloading", f, "...")
    urllib.request.urlretrieve(SNAP_URL + f, path)
    print(f"{f}: {os.path.getsize(path) / 1e6:.1f} MB")

for key, f in files.items():
  print(f"\nFirst lines of {f}:")
  with gzip.open(os.path.join(RAW_DIR, f), "rt") as fh:
    for _ in range(3):
      print("   ", fh.readline().rstrip()[:150])



First lines of wiki-topcats.txt.gz:
    0 10772
    1 2
    1 170193

First lines of wiki-topcats-categories.txt.gz:
    Category:Buprestoidea; 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332
    Category:People_from_Worcester; 1056 1057 1058 1059 1060 60971 76515 76871 78094 79069 79139 79143 80237 84108 84354 84740 85268 85767 89734 90171 953
    Category:Skin_conditions_resulting_from_physical_factors; 971 973 1166 1167 1168 1169 1170 1171 1172 1173 1174 1175 1176 1177 2810 3574 5230 5319 6358

First lines of wiki-topcats-page-names.txt.gz:
    0 Chiasmal syndrome
    1 Kleroterion
    2 Pinakion


In [ ]:
# I import edges file into Pandas tables. I optimize with dtype=np.int32, since
# maximum largest node id (about 1.8 million) can be saved with 32bits.
# I compute number of nodes (largest id amongst source and tagets, +1 because
# first node is 0) and number of edges, and show result.

t0=time.time()
edges=pd.read_csv(os.path.join(RAW_DIR, files["edges"]), sep=r"\s+", header=None,
                         names=["src", "dst"], dtype=np.int32)
N_FULL=int(edges[["src", "dst"]].max().max()) + 1     # largest node id + 1
print(edges.head())
print(f"{N_FULL:,} nodes, {len(edges):,} edges")

# I read categories file line per line and create a dictionary with categories
# names as keys and nodes ids lists as values. I show the number of categories

categories={}
with gzip.open(os.path.join(RAW_DIR, files["categories"]), "rt") as fh:
  for line in fh:
    line= line.strip()
    if line=="":
      continue
    name, ids=line.split(";")
    name=name.replace("Category:", "")
    categories[name]=[int(i) for i in ids.split()]

print(len(categories), "categories")

# I read pages file line per line and create a dictionary with pages (nodes) ids
# as keys and title as values. I show the number of page names (it should match
# with the number of nodes) and how long the code in the cell took.

page_names={}
with gzip.open(os.path.join(RAW_DIR, files["names"]), "rt") as fh:
  for line in fh:
    line=line.rstrip("\n")
    if line=="":
      continue
    idx, title= line.split(" ", 1)
    page_names[int(idx)]=title

print(f"{len(page_names):,} page names  ({time.time() - t0:.0f} s)")

   src     dst
0    0   10772
1    1       2
2    1  170193
3    1  598775
4    2       1
1,791,489 nodes, 28,511,807 edges
17364 categories
1,791,489 page names  (16 s)


In [ ]:
# I collect in a list some scientific keywords and in another list some non
# scientific ones. I check each name of category against the two lists: I'll keep
# just categories that are clearly scientific (not science-fiction, for
# instance). So, I drop categories that are not scientific or that are in the
# excluded keywords.
# If with that category the number of kept nodes would be bigger than 100.000,
# I skip it and go to the next one.
# I collect in a new pandas table the edges where both nodes are in the kept list,
# and I drop self-loops (an article linking to itself) and duplicates.
# Finally I print the number of kept nodes and edges and 20 categories names,
# just to have an idea.

keyw = ["physic", "chemi", "biolog", "mathemat", "astronom", "geolog", "medic",
        "genetic", "zoolog", "botan", "ecolog", "neuro", "scien", "disease",
        "species", "animal", "plant", "insect", "bird", "fish", "anatom",
        "comput", "engineer", "technolog"]
excl = ["fiction", "political", "social", "player", "films", "games", "episode",
        "series", "politician", "people_with", "alternative"]

keep_nodes = set()
keep_categories = []
for name, pages in categories.items():
  if not any(k in name.lower() for k in keyw) or any(x in name.lower() for x in excl):
    continue
  if len(keep_nodes | set(pages)) > 100_000:
    continue
  keep_nodes.update(pages)
  keep_categories.append(name)

print(len(keep_categories), "categories,", len(keep_nodes), "articles")

train_edges = edges[edges["src"].isin(keep_nodes) & edges["dst"].isin(keep_nodes)]
train_edges = train_edges[train_edges["src"] != train_edges["dst"]].drop_duplicates()
print(len(train_edges), "links")
print("\n".join(sorted(keep_categories)[:20]))

560 categories, 82710 articles
586304 links
17th-century_mathematicians
18th-century_mathematicians
19th-century_astronomers
19th-century_mathematicians
20th-century_astronomers
20th-century_mathematicians
21st-century_mathematicians
3D_computer_graphics
Actuarial_science
Aerospace_engineering
Agricultural_pest_insects
Alumni_of_Sciences_Po
American_aerospace_engineers
American_astronomers
American_audio_engineers
American_biochemists
American_biologists
American_civil_engineers
American_computer_businesspeople
American_computer_programmers


In [ ]:
# Import library networkx and create graph G using DiGraph, meaning it is a
# directed graph. I identify componenets and keep the giant one for training.
# Print its size and how many articles are lost.

import networkx as nx
G = nx.from_pandas_edgelist(train_edges, source="src", target="dst",
                            create_using=nx.DiGraph)

Gcomp = nx.weakly_connected_components(G)
G0 = max(Gcomp, key=len)
G_train = G.subgraph(G0).copy()

print(f"{G_train.number_of_nodes():,} nodes, {G_train.number_of_edges():,} links")
print(f"Removed: {len(keep_nodes) - G_train.number_of_nodes():,} articles outside the giant component")

76,960 nodes, 585,980 links
Removed: 5,750 articles outside the giant component


In [ ]:
# Function to describe a graph. It returns some useful metrics.

def describe(G):
  n,m = G.number_of_nodes(), G.number_of_edges()
  d_in = [d for _, d in G.in_degree()]
  d_out = [d for _, d in G.out_degree()]
  scc = max(nx.strongly_connected_components(G), key=len)
  result={"nodes": n, "links": m, "density": nx.density(G),
          "mean degree": m/n, "dangling nodes (%)": 100*sum(d==0 for d in d_out)/n,
          "nodes with in-degree 0 (%)": 100*sum(d==0 for d in d_in)/ n,
          "max in-degree": max(d_in), "largest SCC (% nodes)": 100*len(scc)/n,
          "reciprocity": nx.reciprocity(G),
          "assortativity (in-in)": nx.degree_assortativity_coefficient(G, x="in", y="in")}
  for x in ["in","out"]:
    for y in ["in","out"]:
      result[f"assortativity ({x}-{y})"] = nx.degree_assortativity_coefficient(G, x=x, y=y)
  return result

pd.DataFrame({"training": describe(G_train)})

,training
nodes,"76,960"
links,"585,980"
density,9.89369e-05
mean degree,7.61409
dangling nodes (%),3.81237
nodes with in-degree 0 (%),19.6284
max in-degree,"5,911"
largest SCC (% nodes),69.8051
reciprocity,0.189491
assortativity (in-in),-0.032636


In [ ]:
# The training graph is the giant weakly connected component of the science and
# technology subgraph. It contains 76,960 articles and about 586,000 links,
# so it keeps 92.5% of the 82710 selected articles; the remaining 6,354 were
# isolated or formed small separate groups. The graph is very sparse: its density
# is about 10^{-4}, and each article has on average 7.6 outgoing links inside
# the topic.

# The in-degree is very unequal. About 19.6% of the articles don't receive links from
# the rest of the graph, so that their PageRank takes the smallest value in the
# graph, the same for all of them: they receive only teleportation and their
# share of the dangling nodes; at the other extreme, the most
# linked article receives 5,911 links, from 7.6% of all articles. Only 3.8% of
# the articles are dangling nodes. This proportion is low because wiki-topcats
# was built from the largest strongly connected component of Wikipedia, so every
# article had at least one outgoing link in the full network: the dangling nodes
# of the subgraph are articles whose links point outside the scientific topic,
# so I created them with line
# train_edges = edges[edges["src"].isin(keep_nodes) & edges["dst"].isin(keep_nodes)]
# THey have out degree=0, but indegree is positive, otherwise they wouldn't be
# kept.
# The largest strongly connected component contains 69.8% of the nodes.

# About 19% of the links are reciprocated, that is, the target also links back
# to the source. So, most links go in one direction only.
# Finally, the in-degree assortativity is close to zero (0.03) or "neutral":
# highly linked articles are not more likely than others to receive links
# from highly linked articles. In other words, the in-degree of a node says almost
# nothing about the in-degree of its in-neighbours. This is the information that
# PageRank uses, and that a GNN must learn to obtain from the neighbourhood of
# each node in order to improve on the in-degree baseline.

# The other degree assortativities are also weak (|r|<0.2). The most relevant
# for PageRank is the correlation between the out-degree of the source and the
# in-degree of the target (-0.11): highly linked articles tend to receive their
# links from articles with slightly fewer outgoing links. Since a node passes
# pi(u) / d_out(u) to each of its targets, these links carry a slightly
# larger share of importance, which is one of the ways PageRank can depart from
# in-degree.

In [ ]:
# Create second network, now on geography topic, following the same pattern as with
# first one. Only difference is I exclude nodes already in the scientific graph
# from this one, so that no nodes are shared, as I check at the end.
# Finally show the number of articles and links and some category names.

keyw2 = ["village", "town", "river", "mountain", "island", "lake", "geograph",
         "region", "province", "municipal", "district", "county", "countr",
         "valley", "coast", "volcano", "landform", "populated", "commune",
         "suburb", "neighbourhood", "neighborhood"]

excl2 = ["people", "music", "song", "album", "band", "film", "sport", "football",
         "club", "team", "player", "politic", "election", "mountaineer", "records",
         "school", "faculty"]

keep_nodes2 = set()
keep_categories2 = []
for name, pages in categories.items():
  if not any(k in name.lower() for k in keyw2) or any(x in name.lower() for x in excl2):
    continue
  new = set(pages)-keep_nodes
  if len(keep_nodes2 | new) > 100_000:
    continue
  keep_nodes2.update(new)
  keep_categories2.append(name)

print(len(keep_categories2), "categories,", len(keep_nodes2), "articles")
print("Shared with training:", len(keep_nodes2 & keep_nodes))

second_edges = edges[edges["src"].isin(keep_nodes2) & edges["dst"].isin(keep_nodes2)]
second_edges = second_edges[second_edges["src"] != second_edges["dst"]].drop_duplicates()
print(len(second_edges), "links")
print("Some categories in second graph:")
print("\n".join(sorted(keep_categories2)[:20]))

648 categories, 100000 articles
Shared with training: 0
392561 links
Some categories in second graph:
Active_volcanoes
American_alternative_country_groups
American_geographers
American_municipal_police_officers
Arab_villages_depopulated_during_the_1948_ArabIsraeli_War
Articles_on_towns_and_villages_in_Ireland_possibly_missing_Irish_place_names
Beneov_District
Birds_of_Maraj_Island
Birds_of_the_Yucatn_Peninsula_region
Blansko_District
Borchgrevink_Coast
Bridges_over_the_Mississippi_River
British_Isles_coastal_fauna
Brno-Country_District
Cariboo_Country
Central_Coast_of_British_Columbia
Central_Districts_cricketers
Central_Region,_Uganda
Central_Valley_of_California
Charter_townships_in_Michigan


In [ ]:
# Build the second graph in NetworkX as for the first one and keep its giant
# weakly connected component. Print its statistics next to those of the first.

G2 = nx.from_pandas_edgelist(second_edges, source="src", target="dst",
                            create_using=nx.DiGraph)

Gcomp2 = nx.weakly_connected_components(G2)
G02 = max(Gcomp2, key=len)
G_second = G2.subgraph(G02).copy()

print(f"{G_second.number_of_nodes():,} nodes, {G_second.number_of_edges():,} links")
print(f"Removed: {len(keep_nodes2) - G_second.number_of_nodes():,} articles outside the giant component")
pd.DataFrame({"training": describe(G_train), "second": describe(G_second)})

90,942 nodes, 389,077 links
Removed: 9,058 articles outside the giant component


,training,second
nodes,"76,960","90,942"
links,"585,980","389,077"
density,9.89369e-05,4.70448e-05
mean degree,7.61409,4.2783
dangling nodes (%),3.81237,6.9176
nodes with in-degree 0 (%),19.6284,10.8861
max in-degree,"5,911","7,828"
largest SCC (% nodes),69.8051,71.4499
reciprocity,0.189491,0.379771
assortativity (in-in),-0.032636,-0.0632876


In [ ]:
# The second graph, built in the same way from geography categories, has 90,940
# articles and 389,072 links. Both graphs share the same overall structure: a
# strongly connected core with about 70% of the nodes, a few very highly linked
# articles (maximum in-degree 5,911 and 7,828) and weak degree assortativities
# (|r|<0.2). However, the second graph is sparser (4.3 links per article, against
# 7.6), and its links are reciprocated twice as often (38% against 19%):
# geographic articles often link to each other in both directions, for example
# a village and its municipality. It also has fewer articles with no incoming
# links (11% against 20) and more dangling nodes (6.9% against 3.8%). The second
# graph is therefore a related but not identical network, which makes it a
# demanding test of generalisation.


In [ ]:
# Compute PageRank for both graphs (teleport probability alpha = 0.15; NetworkX's
# alpha is the damping factor, 1 - 0.15 = 0.85) and measure how long it takes.
# PageRank values add up to 1, so their average is 1/N. Multiplying them by the
# number of nodes N makes the average 1 in every graph, so that graphs of
# different size can be compared.
# PageRank has a long tail, since many nodes have very low values and few ones
# have large ones. So we take the natural logarithm, which compresses the scale,
# so that a few large values do not dominate the training loss, and an error then
# means a relative difference. The result, y = log(N * PageRank), is stored in a
# dictionary with the node ids as keys. This is the target that the models will
# learn.

ALPHA = 0.15    # teleport probability

def compute_pagerank(G):
  t0 = time.time()
  pr = nx.pagerank(G, alpha=1 - ALPHA, tol=1e-10, max_iter=1000)
  seconds= time.time()- t0
  N = G.number_of_nodes()
  y = {v: np.log(N * p) for v, p in pr.items()}
  return pr, y, seconds

pr_train, y_train, t_train = compute_pagerank(G_train)
pr_second, y_second, t_second = compute_pagerank(G_second)
print(f"training: {t_train:.1f} s   second: {t_second:.1f} s")

training: 1.5 s   second: 1.0 s


In [ ]:
# Create a pandas defataframe for each graph, with the identification of the node
# (id and title), its in and out degree, its pagerank and its target.
# I do that through a function that I call for G_train and G_second. Then print
# head of these dataframes.

def node_table(G, pr, y):
  nodes= list(G.nodes())
  return pd.DataFrame({"node": nodes,
                       "title": [page_names.get(v, "") for v in nodes],
                       "in_degree": [G.in_degree(v) for v in nodes],
                       "out_degree": [G.out_degree(v) for v in nodes],
                       "pagerank": [pr[v] for v in nodes],
                       "y": [y[v] for v in nodes]})

nodes_train= node_table(G_train, pr_train, y_train)
nodes_second= node_table(G_second, pr_second, y_second)
nodes_train.head()

,node,title,in_degree,out_degree,pagerank,y
0,8,Bing's recognition theorem,1,3,2.11827e-06,-1.81387
1,1101610,3-sphere,21,11,2.12231e-05,0.490623
2,1102709,Annals of Mathematics,142,12,0.000231848,2.88161
3,1102847,R. H. Bing,5,8,1.23918e-05,-0.0474374
4,62,Richard Cytowic,0,5,2.10638e-06,-1.8195


In [ ]:
# I create the 70,15,15 split: I shuffle rows of the train dataframe with frac=1
# and a random seed, in order to reproducibility.
# Then I label 70% of them as train, 15% as "val" and the rest as "test"
# I finally print the value counts.

n = len(nodes_train)
shf= nodes_train.sample(frac=1, random_state=42).index

nodes_train["split"] = "test"
nodes_train.loc[shf[:int(0.70 * n)], "split"] = "train"
nodes_train.loc[shf[int(0.70 * n):int(0.85 * n)], "split"] = "val"
print(nodes_train["split"].value_counts())

split
train    53872
test     11544
val      11544
Name: count, dtype: int64


In [ ]:
# I save in Drive the dataframes and the edges files of both networks in csv
# gzipped files. Then I save the categories of each graph as a text file, which
# records how the graphs were built.

os.makedirs(DATA_DIR, exist_ok=True)

nodes_train.to_csv(f"{DATA_DIR}/train_nodes.csv.gz", index=False)
nodes_second.to_csv(f"{DATA_DIR}/second_nodes.csv.gz", index=False)
nx.to_pandas_edgelist(G_train).to_csv(f"{DATA_DIR}/train_edges.csv.gz", index=False)
nx.to_pandas_edgelist(G_second).to_csv(f"{DATA_DIR}/second_edges.csv.gz", index=False)

with open(f"{DATA_DIR}/categories_train.txt", "w") as f:
  f.write("\n".join(sorted(keep_categories)))
with open(f"{DATA_DIR}/categories_second.txt", "w") as f:
  f.write("\n".join(sorted(keep_categories2)))

In [ ]:
# A simple way to predict PageRank for the nodes of a graph is a linear
# regression: each training node is a point in a cartesian plane
# with x=log(1+in-degree) and y= log(N*pi), where pi is PageRank and N the number
# of nodes. We fit a linear regression model and there we have, a function that
# predicts PageRank (or log(N*pi)) from the in-degree: it will be a straight line
# closest to these points.
# This is the baseline: if a GNN does not rank the nodes better (Spearman, top-n%,
# NDCG), the extra effort is not worth it, because the model has learnt nothing
# beyond counting incoming links.
# The code fits the regression, prints the equation of the line, and stores the
# predictions for all nodes of both graphs in a new column of the dataframes,
# pred_baseline.

tr=nodes_train[nodes_train["split"]=="train"]
baseline= LinearRegression().fit(np.log1p(tr[["in_degree"]]), tr["y"])
print(f"y = {baseline.intercept_:.3f} + {baseline.coef_[0]:.3f} * log(1 + in-degree)")

nodes_train["pred_baseline"]= baseline.predict(np.log1p(nodes_train[["in_degree"]]))
nodes_second["pred_baseline"] = baseline.predict(np.log1p(nodes_second[["in_degree"]]))

y = -1.919 + 0.790 * log(1 + in-degree)


In [ ]:
# Function that computes the metrics for a set of nodes: it receives as
# parameters a dataframe and the name of the column with a model's predictions.
# It computes and returns:
# 1) The Spearman correlation between the predicted and the true ranking.
# 2) The MSE (Mean Squared Error), which is the average of the squared
#    difference between the prediction and the target.
# 3) For p=1,5 and 10, the proportion of the true top p% nodes that are
#    also in the predicted top p%.
# 4) NDCG@p: for p=1,5 and 10, looks at the p% nodes the model ranks highest, in
#    the prediction order. Each node has its true PageRank (N*pi) and it's
#    multiplied by a weight that decreases with its position (1 for the first,
#    0.63 for the second, 0.5 for the third…). The total is divided by the score
#    of the true ranking, so NDCG is 1 when the order matches the true one.
#    Unlike top-p%, it rewards placing the most important nodes at the very top,
#    and it penalises missing a hub more than missing a minor node.
#    ( https://medium.com/@mukulranjan/understanding-ndcg-885656321b3b )
# For top-p% and NDCG: If several nodes have the same prediction and not all of
# them fit in the top, the ones that appear first in the table are chosen.


def metrics(df, pred):
  res = {"spearman": spearmanr(df["y"], df[pred])[0],
         "MSE": ((df[pred] - df["y"]) ** 2).mean()}
  for p in [1, 5, 10]:
    k=max(1, round((p/100)*len(df))) # how many are the top-p%
    true_top=df.nlargest(k, "y") # top-p% nodes in reality, based on y
    pred_top=df.nlargest(k, pred) # top-p% nodes based on prediction
    true_ids=set(true_top["node"]) # ids of the real top-p%
    pred_ids=set(pred_top["node"]) # ids of the predicted top-p%
    common=true_ids & pred_ids # ids that are both real and predicted
    res[f"top-{p}%"]=len(common)/k # number of common relative to size of the top, so that it's a proportion
    # NDCG: like top-p%, but position matters and important nodes count more.
    pred_values=np.exp(pred_top["y"]).to_numpy()   # TRUE N*pi of the predicted top, in predicted order
    true_values=np.exp(true_top["y"]).to_numpy()   # TRUE N*pi of the real top, in the best order
    dcg=0
    ideal=0
    for i in range(k):    # i = 0, 1, ..., k-1 => position= i + 1
      weight= 1/np.log2(i+2) # 1 for 1st place, 0.63 for 2nd, 0.5 for 3rd, ...
      dcg+=pred_values[i]*weight
      ideal+=true_values[i]*weight
    res[f"NDCG@{p}%"] = dcg/ideal  # 1 = perfect order
  return res

In [ ]:
# Metrics of the baseline on each set of nodes: train, val and test nodes of the
# training graph, and all nodes of the second graph. Saved to Drive.

baseline_metrics={}
for s in ["train", "val", "test"]:
    baseline_metrics[f"training graph, {s} nodes"] = metrics(nodes_train[nodes_train["split"] == s], "pred_baseline")
baseline_metrics["second graph, all nodes"] = metrics(nodes_second, "pred_baseline")

table = pd.DataFrame(baseline_metrics).T
table.to_csv(f"{DATA_DIR}/baseline_metrics.csv")
table

,spearman,MSE,top-1%,NDCG@1%,top-5%,NDCG@5%,top-10%,NDCG@10%
"training graph, train nodes",0.848172,0.338612,0.669759,0.893827,0.682257,0.906397,0.679785,0.912085
"training graph, val nodes",0.84596,0.336329,0.704348,0.881422,0.688042,0.900038,0.67851,0.906129
"training graph, test nodes",0.850955,0.342764,0.73913,0.871649,0.682842,0.888044,0.684575,0.896182
"second graph, all nodes",0.769327,0.386936,0.676568,0.908679,0.602155,0.897133,0.583132,0.894567


In [ ]:
# Function that creates a perturbation of a fraction frac of links of graph G,
# with a random seed.
# It deletes and creates the same number of edges, at random. When creating,
# it checks that edge doesn't already exist. It returns the perturbed new graph.

def perturb(G, frac, seed):
  """Copy of G with frac of its links changed: half deleted, half added at random."""
  rng=random.Random(seed)
  H=G.copy()
  k=round(frac*G.number_of_edges()/2) # number of links to delete, and to add

  # Delete k random links
  to_delete= rng.sample(list(H.edges()), k)
  H.remove_edges_from(to_delete)

  # Add k random new links: two different nodes, not linked in the original graph
  nodes=list(H.nodes())
  add=0
  while add<k:
    u=rng.choice(nodes)
    v=rng.choice(nodes)
    if u!=v and not G.has_edge(u, v) and not H.has_edge(u,v):
      H.add_edge(u, v)
      add+=1
  return H

In [ ]:
# Creation of the perturbed graphs of train, with 1,5,10 and 20% perturbation.
# Compute their pagerank, create a dataframe with pagerank and target for each
# node and merge with the non-perturbed graph dataframe, to get the split and
# the original target (y_old).
# This allows for the test nodes, to compare the new (true) PageRank with two
# cheap predictions: the in-degree baseline on the new graph, and the old PageRank
# values, as if we had not recomputed after the change.
# Save nodes and edges as compressed CSV files and show the metrics in a table.

LEVELS=[0.01, 0.05, 0.10, 0.20]
old=nodes_train[["node", "split", "y"]].rename(columns={"y": "y_old"})
perturbed_metrics={}

for fr in LEVELS:
  level=int(fr * 100)
  H= perturb(G_train, fr, seed=42 + level)
  pr_H,y_H,t_H=compute_pagerank(H)

  nodes_H= node_table(H, pr_H, y_H).merge(old, on="node")     # add split and old target
  nodes_H["pred_baseline"]= baseline.predict(np.log1p(nodes_H[["in_degree"]]))
  test_H= nodes_H[nodes_H["split"]=="test"]

  perturbed_metrics[f"{level}%, baseline"] = metrics(test_H, "pred_baseline")
  perturbed_metrics[f"{level}%, old PageRank"] = metrics(test_H, "y_old")

  nodes_H.to_csv(f"{DATA_DIR}/perturbed_{level:02d}_nodes.csv.gz", index=False)
  nx.to_pandas_edgelist(H).to_csv(f"{DATA_DIR}/perturbed_{level:02d}_edges.csv.gz", index=False)
  print(f"{level}%: {H.number_of_edges():,} links, PageRank in {t_H:.1f} s")

table=pd.DataFrame(perturbed_metrics).T
table.to_csv(f"{DATA_DIR}/perturbed_metrics.csv")
table

1%: 585,980 links, PageRank in 4.4 s
5%: 585,980 links, PageRank in 1.0 s
10%: 585,980 links, PageRank in 0.9 s
20%: 585,980 links, PageRank in 1.0 s


,spearman,MSE,top-1%,NDCG@1%,top-5%,NDCG@5%,top-10%,NDCG@10%
"1%, baseline",0.848402,0.340989,0.730435,0.872394,0.682842,0.888527,0.682842,0.896725
"1%, old PageRank",0.98905,0.00838712,0.973913,0.999507,0.991334,0.999667,0.987002,0.999461
"5%, baseline",0.841947,0.328492,0.756522,0.885662,0.682842,0.896907,0.686308,0.903691
"5%, old PageRank",0.958203,0.035352,0.973913,0.996726,0.960139,0.996292,0.958406,0.995989
"10%, baseline",0.827578,0.316428,0.765217,0.884913,0.682842,0.89374,0.695841,0.904468
"10%, old PageRank",0.910989,0.0806484,0.956522,0.990438,0.963605,0.99032,0.931542,0.989188
"20%, baseline",0.815701,0.297825,0.782609,0.876418,0.679376,0.886925,0.697574,0.897676
"20%, old PageRank",0.836254,0.173713,0.947826,0.987185,0.930676,0.985071,0.898614,0.983885
